# Create subplot-level summary statistics from tree inventory data
John L. Godlee (johngodlee@gmail.com)  
Last updated: 2025-06-18

In [ ]:
# Define directories
indir <- "../dat/clean/panama"
outdir <- "../dat/processed/panama"

In [ ]:
# Packages
library(dplyr)
library(sf)
library(BIOMASS)

In [ ]:
# Source functions
source("./func.R")

In [ ]:
# Import stem data
s <- read.csv(file.path(indir, "stems_clean.csv"))

In [ ]:
# Import plot polygons
p <- read_sf(file.path(indir, "polys.gpkg"))

In [ ]:
# Import plot origins
o <- read_sf(file.path(indir, "origins.gpkg"))

In [ ]:
ps <- read_sf(file.path(outdir, "subplot_polys.gpkg"))

In [ ]:
# Convert stem data to sf object
s_sf <- s %>% 
  st_as_sf(., coords = c("x_grid", "y_grid"))

In [ ]:
# Split stem data by plot
s_split <- split(s_sf, s_sf$plot_id)[p$plot_id]

In [ ]:
# Rotate stem coordinates from each plot
s_trans <- globalCoord(s_split, p, o, p$bearing, name = colnames(s_sf))

In [ ]:
# Split subplots by plot 
ps_split <- split(ps, ps$plot_id)[p$plot_id]

In [ ]:
# Assign stems to subplots 
s_sub <- subplotAssign(s_trans, ps_split, 
  name = c(colnames(s_sf), "subplot_id"))

In [ ]:
# Join stem dataframes
s_all <- bind_rows(s_sub)

In [ ]:
# TODO: Move code below here to individual files when methods developed
# Get wood density 
s_all$wd <- getWoodDensity(
  genus = s_all$genus, 
  species = s_all$species,
  stand = s_all$plot_id,
  family = s_all$family,
  region = "SouthAmericaTrop")$meanWD

In [ ]:
# Add plot mean coordinates to stem data
p_cent <- p %>% 
  st_centroid(.) %>% 
  st_transform(., 4326) %>% 
  bind_cols(., st_coordinates(.)) %>% 
  dplyr::select(
    plot_id,
    plot_centroid_x = X,
    plot_centroid_y = Y) %>% 
  st_drop_geometry()

In [ ]:
s_c <- s_all %>% 
  left_join(., p_cent, by = "plot_id")

In [ ]:
# Estimate stem biomass
s_c$agb <- computeAGB(
  D = s_c$diam,
  WD = s_c$wd,
  H = NULL,
  coord = st_drop_geometry(s_c[,c("plot_centroid_x", "plot_centroid_y")]),
  Dlim = 5)

In [ ]:
# Summarise biomass to subplot level
s_summ <- s_c %>% 
  st_drop_geometry() %>% 
  left_join(., 
    st_drop_geometry(ps)[,c("plot_id", "subplot_id", "area")], 
    by = c("plot_id", "subplot_id")) %>% 
  mutate(ba = (pi * (diam/2)^2) / 10000) %>% 
  subplotSumm(., c("site_id", "plot_id", "subplot_id"), "area", "agb", "ba", "wd")

In [ ]:
# Write summarised data to file
write.csv(s_summ, file.path(outdir, "subplot_summ.csv"), row.names = FALSE)